# Time-Independent Schrödinger Equation (TISE) Simulation

Update history
- 2021.01.01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code. (v1)
- 2024.06.26 : Minsu Jeong,  KAIST Electrical Engineering, minor revision and clean up
- 2025.03.27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization and the flexibility of the shape of the potential. (v2)
- 2025.09.01 : Minsu Jeong,  KAIST Electrical Engineering, revision and clean up
- 2026.09.03 : Seungho Chung, updated the entire codebse for python 3.14 version
- 2026.09.17 : Sparse low-state solver, 80 Å geometry-based box, bound-state selection, and automatic plot ranges.

ref
1. D. J. Griffiths and D. F. Schroeter, *Introduction to Quantum Mechanics*, 3rd ed., Cambridge University Press (2018). [Publisher](https://www.cambridge.org/highereducation/books/introduction-to-quantum-mechanics/990799CA07A83FC5312402AF6860311E).
2. D. A. Neamen, *Semiconductor Physics and Devices: Basic Principles*, 4th ed., McGraw-Hill (2012), p. 31.
3. John R. Hiller, *Quantum Mechanics Simulations*, The Consortium for Upper-Level Physics Software, John Wiley & Sons.
4. Bengt Fornberg, *Generation of finite difference formulas on arbitrarily spaced grids*, Mathematics of Computation **51**, 699–706 (1988). [DOI](https://doi.org/10.1090/S0025-5718-1988-0935077-0). Background on finite-difference weights; this notebook retains its original coefficient generator.
5. SciPy, [`scipy.sparse.linalg.eigsh`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html).
6. Plotly, [Axes in Python](https://plotly.com/python/axes/).

---

## Usage Guide

### Run a calculation

1. Execute the **Calculation** code cell, then the **Visualization & interactive widget** code cell.
2. Select **Potential Shape**, **Effective mass**, and the available well parameters.
3. Set **# of States**, then click **Run Interact**. Changing a control alone does not redraw the results.

The output contains the actual box length and grid spacing, energy components for the selected states, and a potential/eigenstate plot. Positions are displayed in angstroms (Å), energies in eV. **# of States** is an upper limit: fewer curves appear when fewer calculated states pass the selection cutoff.

### Parameters and controls

For finite square and asymmetric wells, the well bottom is zero and the exterior barrier is $H>0$.

| Symbol | Control or code setting | Meaning and unit |
| --- | --- | --- |
| $H$ | **Height [eV]** | Exterior barrier height for finite square wells, in eV |
| $w$ | **Width [Ang]** | One well's width; triangular support width, in Å |
| $d$ | **Distance [Ang]** | Barrier-region width between the two wells, in Å |
| $m_{\rm eff}$ | **Effective mass** | Dimensionless mass ratio $m^*/m_e$: 0.07, 0.55, 1.00, or 1.08 |
| $c$ | **Curvature of harmonic well** | Dimensionless multiplier of the existing box-normalized parabola |
| $\alpha$ | **Slope of triangular well [eV/Ang]** | Potential slope inside the triangular region, in eV/Å |
| $n$ | **# of States** | Maximum number of low states to display, from 1 to 10; default 5 |
| $P$ | `BOX_PADDING_ANG = 80.0` | Extra space outside the well region, in Å |
| $\Delta x$ | `GRID_SPACING_ANG` | Spatial resolution, approximately 0.20 Å |
| $L$ | Calculated box length | Determined from the potential geometry and padding |

Only controls relevant to the selected shape are shown. The current Height value also sets the wavefunction display scale and some retained display cutoffs, including when the Height control is hidden.

| Potential Shape | Well parameters | Box construction |
| --- | --- | --- |
| **Finite square well** | Width, Height | One centered well; padding on both sides |
| **Infinite square well** | Width | One centered well with large finite exterior walls; padding on both sides |
| **Double square well** | Width, Distance, Height | Two centered wells; padding outside both wells |
| **Harmonic oscillator** | Curvature | Existing 500 Å box and normalized parabola |
| **Triangular well** | Width, Slope | Finite rising segment; padding on both sides |
| **Asymmetric square well (1)** | Width, Height | One well next to the left wall; padding on the right |
| **Asymmetric square well (2)** | Width, Distance, Height | Two wells next to the left wall; padding on the right |
| **Custum potential** | Existing editable profile | Existing 500 Å box; currently flat inside the boundary walls |

The **Custum potential** label is retained from the original controls. Widths and separations are discretized to grid intervals, so the represented geometry can differ slightly from the input length. In asymmetric modes, the leftmost boundary-wall points also reduce the free well region.

### Set the extra space around the wells

The approximate geometric box lengths are

$$L\simeq w+2P\quad\text{(centered single well)},$$
$$L\simeq2w+d+2P\quad\text{(centered double well)},$$
$$L\simeq w+P\quad\text{or}\quad2w+d+P\quad\text{(asymmetric wells)}.$$

The code rounds to grid intervals and reserves additional space for the boundary-wall stencil. It retains the same spatial resolution as the box size changes. Harmonic and Custom potentials keep their original domain because their present definitions do not provide an independent well-support length.

To change the margin, edit `BOX_PADDING_ANG` in the Calculation cell and rerun both code cells. The default 80 Å favors quick exploration. States very close to the exterior barrier can extend well beyond this margin and may disappear from the finite-box bound-state selection; increase the margin when examining these states.

### Read the plot and energy components

- **Potential:** the black curve is $V(x)$.
- **Re(Psi) state j:** red curves show the real part of each wavefunction, shifted by its eigenenergy. Red shades vary with state rank.
- **Im(Psi) state j:** blue curves show the imaginary part, with the same energy offset and rank-dependent shades.
- **|Psi|² state j:** gray shading shows a scaled probability distribution above the true energy baseline.
- Energy labels show the actual eigenenergies. The height of a wavefunction or shaded curve is an arbitrary display amplitude, rather than an additional energy.
- The printed **Kinetic**, **Potential**, and **Total** values are expectation values in the same eigenstate. Total agrees with the eigenenergy to numerical accuracy; printed values are rounded.

The horizontal range follows the displayed states and well regions. The vertical range includes the displayed amplitudes and densities, using one range throughout the animation. Large artificial walls can extend above the visible range. Plotly zoom, pan, and legend controls remain available; manual x/y range sliders are removed.

The default output is stationary. The code also supports a time phase for each eigenstate; the existing time-duration widget is currently hidden in the layout. Individual-state probability densities stay constant while the real and imaginary parts oscillate.

## Basic Theory

### An electron in a one-dimensional potential

The time-independent Schrödinger equation determines an eigenenergy $E_j$ and wavefunction $\psi_j(x)$:

$$H\psi_j=E_j\psi_j,\qquad H=-\frac{\hbar^2}{2m^*}\frac{d^2}{dx^2}+V(x).$$

The potential describes the well geometry; the kinetic term depends on the spatial variation of the wavefunction. The calculation samples this equation on a uniform grid and solves a matrix eigenvalue problem for a limited number of low states. [1–3]

### Finite wells and bound states

For the finite square, double and asymmetric wells, $V=0$ inside a well and $V=H$ in the exterior region. A bound state satisfies $E_j<H$ and decays in that exterior region. States near $H$ have longer tails and require more space around the well.

The calculation has a finite box with large potentials at both ends. Its spectrum is discrete even above the exterior barrier. The finite-well display selects only states below that barrier; the number shown depends on both the requested count and this cutoff. Infinite, harmonic, triangular and Custom shapes have the selection rules listed in **Code Implementation**.

### Closed boundaries and the energy reference

Ideal hard walls impose a zero wavefunction at the endpoints. This implementation represents them using very large finite potentials on the first and last three grid points, together with the original periodic finite-difference stencil. Low-energy states have strongly suppressed amplitudes in those wall regions.

The energy zero is the bottom of the square wells. The printed and plotted energies retain that reference; the numerical shift used by the eigensolver does not change it.

### Wavefunctions and stationary probability

A stationary eigenstate evolves by a phase,

$$\psi_j(x,t)=\psi_j(x,0)e^{-iE_jt/\hbar}.$$

Its real and imaginary parts can change with time, while $|\psi_j(x,t)|^2$ is constant. An overall sign or constant phase is arbitrary and preserves energy and probability density. In degenerate subspaces, different orthonormal combinations can represent the same energy. [1–3]

<details>
<summary><strong>Advanced Theory</strong></summary>

<h3>Spatial grid and the second-derivative matrix</h3>

<p>Let $N$ be the number of grid points, $\Delta x$ the spacing, and $L=N\Delta x$ the box length. The displayed grid has no duplicate endpoint:</p>

$$x_i=\left(i-\frac{N-1}{2}\right)\Delta x,\qquad i=0,\ldots,N-1.$$

<p>$D_2$ is the matrix approximating the second spatial derivative. The default <code>laplacian_order = 3</code> uses the seven-point stencil</p>

$$[D_2\boldsymbol\phi]_i=\frac{1}{\Delta x^2}\left[-\frac{49}{18}\phi_i+\frac32(\phi_{i-1}+\phi_{i+1})-\frac3{20}(\phi_{i-2}+\phi_{i+2})+\frac1{90}(\phi_{i-3}+\phi_{i+3})\right].$$

<p>The coefficient generator retains its original moment-condition calculation; Fornberg provides background on finite-difference weights. Stencil indices wrap modulo $N$, and the potential supplies the large end walls. The smooth-function sixth-order stencil accuracy does not imply sixth-order eigenenergy accuracy for discontinuous square wells. [4]</p>

<h3>Effective atomic units</h3>

<p>With $m_{\rm eff}=m^*/m_e$, the internal length and energy units are</p>

$$a^*=\frac{a_0}{m_{\rm eff}},\qquad E_h^*=m_{\rm eff}E_h.$$

<p>Input Å lengths and eV energies are converted to these units before constructing the Hamiltonian. The internal matrix is</p>

$$H=-\frac12D_2+\operatorname{diag}(V).$$

<p>Multiplying its eigenenergies by <code>effective_har2ev</code> gives eV. Animation phases use these energies converted to joules and time converted to seconds. The mass dependence is already included in the effective-unit conversions.</p>

<h3>Sparse Hamiltonian and shift–invert eigenvalues</h3>

<p>The Laplacian has at most seven entries per row, and the potential is diagonal. Both are stored as CSC sparse matrices, giving $O(N)$ matrix storage for the fixed stencil. <code>eigsh</code> computes only the requested low eigenpairs rather than the entire spectrum. [5]</p>

<p>For the usual single energy group, the solver chooses $\sigma=\min(V)-1$ in effective Hartree units and applies shift–invert:</p>

$$(H-\sigma I)^{-1}\boldsymbol\phi_j=\frac{1}{E_j-\sigma}\boldsymbol\phi_j.$$

<p>The kinetic matrix is positive semidefinite, so this shift is below the spectrum. Selecting the largest transformed magnitudes finds the lowest original energies. Sparse factorization applies the inverse through linear solves. Returned energies remain unshifted. The existing separate-shift logic is retained for strongly separated potential groups. [5]</p>

<p>The requested count is capped by the number of potential-grid values below the selection cutoff. Since $H=T+V$ and $T\geq0$, its ordered eigenvalues cannot be below the corresponding ordered diagonal-potential values. This gives an upper bound on how many states can pass the cutoff. It does not determine the exact bound-state count.</p>

<p>Energies are sorted in ascending order, and the same permutation is applied to eigenvector columns. Equal energies remain separate states; their vectors can rotate within a degenerate eigenspace.</p>

<h3>Potential geometry and finite boundaries</h3>

<p>The square profiles use integer-grid slices. Double-well Distance is the barrier interval between the two well regions. Asymmetric wells start next to the left box boundary; the first three points are overwritten by the end-wall potential after the shape is assembled.</p>

<p>The harmonic profile retains its existing normalized definition:</p>

$$V_i=Hc\left(\frac{i-i_c}{i_c}\right)^2,\qquad i_c=\frac{N-1}{2},$$

<p>for the interior of the retained odd-point grid. Here $c$ is dimensionless. Changing the box would change the physical curvature at fixed $H,c$, so this mode retains its original 500 Å domain. The triangular segment uses $V_i=\alpha(x_i-x_{\rm left})$ inside its finite support and a large exterior plateau.</p>

<p>The end-wall value is $10^9$ effective Hartree; the triangular exterior plateau is $10^6$ effective Hartree. These are numerical potentials, not literal infinity. Their physical eV values depend on the selected mass.</p>

<h3>Normalization and displayed amplitudes</h3>

<p>The stored eigenvector $\boldsymbol\phi_j$ satisfies $\sum_i|\phi_{ij}|^2=1$. In displayed Å coordinates, a continuous density is represented by $|\phi_{ij}|^2/\Delta x_{\rm Å}$. The plot uses discrete amplitudes and probabilities for visualization rather than this continuous-density normalization.</p>

<p>Define the display scale $S=0.8H$ in eV and $\phi_{ij}(t)=\phi_{ij}e^{-iE_jt/\hbar}$. For unit-normalized solver vectors, the plotted curves are</p>

$$y_{\rm real}=E_j+S\operatorname{Re}\phi_{ij}(t),\qquad y_{\rm imag}=E_j+S\operatorname{Im}\phi_{ij}(t),$$

$$y_{\rm density}=E_j+6S|\phi_{ij}|^2.$$

<p>The factors $S$ and $6S$ make the curves visible on an energy axis. They do not change the eigenenergies. Display amplitudes depend on the discrete normalization and therefore on grid spacing.</p>

<h3>Automatic spatial and energy ranges</h3>

<p>The spatial range includes approximately 99.9% of each displayed state's probability, all finite well regions, and a margin. This changes only the view; it preserves stored vectors and the state cutoff.</p>

<p>The energy range includes the potential bottom and the envelopes</p>

$$E_j-S\max_i|\phi_{ij}|\quad\text{through}\quad E_j+\max\left(S\max_i|\phi_{ij}|,\,6S\max_i|\phi_{ij}|^2\right).$$

<p>These envelopes include real/imaginary excursions at any animation phase and the density curve. One range is used throughout the animation. Artificial walls and plateaus may extend above it, keeping the displayed states readable. If no state passes the cutoff, the range is based on the potential region. [6]</p>

</details>

<details>
<summary><strong>Code Implementation</strong></summary>

<h3>Calculation flow and entry points</h3>

<p>The <strong>Run Interact</strong> button calls <code>main</code> with the current control values. The two code cells keep their original order: Calculation defines units and numerical operators; Visualization defines the plotting functions and widgets.</p>

<table>
<thead><tr><th>Stage</th><th>Functions or settings</th><th>Result</th></tr></thead>
<tbody>
<tr><td>Units and settings</td><td><code>UnitSystem</code>, <code>SimulationConfig</code>, <code>BOX_PADDING_ANG</code>, <code>GRID_SPACING_ANG</code></td><td>Effective units, padding and spatial resolution</td></tr>
<tr><td>Box and potential</td><td><code>main</code> → <code>simulation_grid</code> → <code>potential</code></td><td>Current box size and grid potential</td></tr>
<tr><td>Hamiltonian</td><td><code>laplacianCoeff</code> → <code>laplacian</code> → <code>hamiltonian</code></td><td>Real symmetric sparse operator</td></tr>
<tr><td>Low eigenpairs</td><td><code>solve_tise</code>, <code>state_energy_cutoff</code></td><td>Ascending energies and corresponding vector columns</td></tr>
<tr><td>Energy components</td><td><code>compute_energy</code></td><td>Kinetic, potential and total expectation values in eV</td></tr>
<tr><td>Plot and animation</td><td><code>animate</code> → <code>state_plot_ranges</code></td><td>Potential and state curves with automatic ranges</td></tr>
</tbody>
</table>

<h3>Box size and potential assembly</h3>

<p><code>simulation_grid</code> returns <code>(boxL_ang, ngridx, dx_ang)</code>. It matches the integer-grid well geometry, adds padding on both sides for centered wells or only on the right for asymmetric wells, and reserves space for the end-wall stencil. Harmonic and Custom retain <code>SIM_CFG.boxL_ang</code> and <code>SIM_CFG.ngridx</code>.</p>

<p><code>main</code> converts lengths, heights and slopes to effective atomic units. <code>potential</code> assembles the selected profile and applies the end walls last. <code>laplacian</code> builds COO row/column entries and converts to CSC; <code>hamiltonian</code> adds the sparse diagonal potential.</p>

<h3>Low-state calculation and selection cutoffs</h3>

<p><code>solve_tise(..., num_state=n)</code> returns energy and vector arrays of shapes <code>(k,)</code> and <code>(ngridx, k)</code>, with $k\leq n$. Its potential-based cap can reduce the computed count. The final energy cutoff is applied when printing and plotting, so the number displayed can be smaller than $k$.</p>

<table>
<thead><tr><th>Potential</th><th>Energy selection</th><th>Meaning</th></tr></thead>
<tbody>
<tr><td>Finite, double, asymmetric square wells</td><td>$E&lt;H$</td><td>Bound states below the exterior barrier</td></tr>
<tr><td>Triangular well</td><td>$E&lt;10^6$ effective Hartree</td><td>States below the artificial exterior plateau</td></tr>
<tr><td>Infinite square well</td><td>$E&lt;100H$</td><td>Retained display cap; the exterior potential is much larger</td></tr>
<tr><td>Harmonic and Custom</td><td>$E&lt;1.5H$</td><td>Retained display caps</td></tr>
</tbody>
</table>

<p><code>state_energy_cutoff</code> supplies the same rule to the solver, <code>compute_energy</code> and <code>animate</code>. Cutoffs are evaluated in effective units internally and converted consistently to eV for display.</p>

<h3>Energy components and plots</h3>

<p><code>compute_energy</code> reconstructs the same sparse kinetic and potential operators and evaluates $\langle T\rangle$, $\langle V\rangle$ and their sum for the selected vectors.</p>

<p><code>animate</code> builds the same uniformly spaced position grid, applies the shared cutoff, and creates real, imaginary and density-fill traces. State labels are one-based and refer to the selected low-energy order. Red/blue shades and gray density shading retain the existing conventions.</p>

<p><code>state_plot_ranges</code> derives x limits from cumulative probability and well regions, then derives y limits from true energies and phase-independent display envelopes. Energy labels are positioned inside those ranges. No x/y range widgets are created.</p>

<p><code>update_visibility</code> shows only the shape-specific controls. <code>interact_manual</code> applies settings on the button click. The existing <code>Simulation Time [fsec]</code> widget remains hidden and defaults to zero; <code>animate</code> supports time evolution when supplied a positive duration. Plotly time controls appear only for positive duration.</p>

</details>


---
# (1) Calculation

In [1]:
"""
Update history
2021. 01. 01 : Hyeonwoo Yeo, KAIST Electrical Engineering, Initial implementation of TISE code.
2024. 06. 26 : Minsu Jeong,  KAIST Electrical Engineering, minor revision and clean up
2025. 03. 27 : Minsu Jeong,  KAIST Electrical Engineering, updated the vidualization and the flexibility of the shape of the potential.
2025. 09. 01 : Minsu Jeong,  KAIST Electrical Engineering, revision and clean up
2026.09.17 : Sparse low-state solver, 80 Å geometry-based box, bound-state selection, and automatic plot ranges.

Reference :
- Griffith, Introduction to Quantum Mechanics, 3ed (2018)
- Neaman, Semiconductor Physics and Devices Basic Principles, 4ed (McGraw-Hill, 2012), p31
- John R. Hiller, Quantum Mechanics Simulations The Consortium for upper-Level Physics Software, (John Wiley & Sons)
- Bengt Fornberg, Generation of finite difference formulas on arbitrarily spaced grids, Math. Comp. 51, 699-706 (1988), https://doi.org/10.1090/S0025-5718-1988-0935077-0
- SciPy, scipy.sparse.linalg.eigsh: https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.linalg.eigsh.html
- Plotly, Axes in Python: https://plotly.com/python/axes/
"""


###############################################################################
# Imports and Configuration
###############################################################################

import numpy as np
import numpy.linalg as lin
import math
from dataclasses import dataclass
import ipywidgets as widgets
from ipywidgets import interact_manual
from IPython.display import HTML, display
import plotly.graph_objects as go
import plotly.express as px
from scipy.sparse import coo_matrix, diags
from scipy.sparse.linalg import eigsh
from scipy import constants as sc_const
from scipy.constants import physical_constants

# Set numpy print options for debugging (optional)
np.set_printoptions(threshold=784, linewidth=np.inf)

###############################################################################
# Constants and Unit Conversions
###############################################################################

# Constant and unit conversion
@dataclass(frozen=True)
class UnitSystem:
    hbar: float
    bohr_radius_m: float
    angstrom_m: float
    hartree_energy_j: float
    electron_volt_j: float
    ang2bohr: float
    har2ev: float


@dataclass
class SimulationConfig:
    boxL_ang: float = 500.0
    ngridx: int = 2501
    laplacian_order: int = 3


def _build_unit_system():
    bohr_radius_m = sc_const.physical_constants["Bohr radius"][0]
    angstrom_m = sc_const.angstrom
    hartree_energy_j = sc_const.physical_constants["Hartree energy"][0]
    electron_volt_j = sc_const.physical_constants["electron volt"][0]
    return UnitSystem(
        hbar=sc_const.physical_constants["reduced Planck constant"][0],
        bohr_radius_m=bohr_radius_m,
        angstrom_m=angstrom_m,
        hartree_energy_j=hartree_energy_j,
        electron_volt_j=electron_volt_j,
        ang2bohr=angstrom_m / bohr_radius_m,
        har2ev=hartree_energy_j / electron_volt_j,
    )


UNITS = _build_unit_system()
SIM_CFG = SimulationConfig()
BOUNDARY_WALL_HAR = 1e9
BOX_PADDING_ANG = 80.0                 # Extra space outside the well [Angstrom]
GRID_SPACING_ANG = SIM_CFG.boxL_ang / SIM_CFG.ngridx  # Keep the existing resolution

# Unit aliases used by solver/plot functions in later cells
hbar = UNITS.hbar
bohr_radius = UNITS.bohr_radius_m       # in meter
angstrom_radius = UNITS.angstrom_m      # in meter
hartree_energy = UNITS.hartree_energy_j # in Joule
ev_energy = UNITS.electron_volt_j       # in Joule
ang2bohr = UNITS.ang2bohr               # ~1.88973
har2ev = UNITS.har2ev                   # ~27.21140795


###############################################################################
# Grid Setting
###############################################################################

def simulation_grid(pot_shape, width_ang, distance_ang, m_eff=1.0):
    """Size the box from the well geometry, keeping the grid spacing fixed."""
    if (not np.all(np.isfinite([width_ang, distance_ang, m_eff, BOX_PADDING_ANG]))
            or width_ang <= 0 or distance_ang < 0 or m_eff <= 0 or BOX_PADDING_ANG < 0):
        raise ValueError("Use a positive well width/mass and nonnegative distance/padding.")
    if not np.isfinite(GRID_SPACING_ANG) or GRID_SPACING_ANG <= 0:
        raise ValueError("Grid spacing must be positive.")
    order = SIM_CFG.laplacian_order
    if pot_shape in (3, 7):
        # Harmonic curvature depends on the box; Custom has no separate well width.
        # Retain their existing domain and potential definitions.
        return SIM_CFG.boxL_ang, SIM_CFG.ngridx, SIM_CFG.boxL_ang / SIM_CFG.ngridx
    if pot_shape not in (0, 1, 2, 4, 5, 6):
        raise ValueError("Unknown potential shape.")
    conversion = ang2bohr * m_eff
    dx = GRID_SPACING_ANG * conversion
    width = int(width_ang * conversion / dx)
    distance = int(distance_ang * conversion / dx)
    padding = int(np.ceil(BOX_PADDING_ANG / GRID_SPACING_ANG))
    # Match the discretized well widths used by potential().
    if pot_shape in (0, 1):
        core = 2 * (width // 2)
    elif pot_shape == 2:
        core = 2 * width + 2 * (distance // 2)
    elif pot_shape == 4:
        core = max(2, width)
    elif pot_shape == 5:
        core = width
    else:
        core = 2 * width + distance
    if pot_shape in (5, 6):
        # Keep the physical left wall; add space only on the right.
        ngridx = core + padding + order
    else:
        # Add padding on both sides plus room for the boundary-wall stencil.
        ngridx = core + 2 * padding + 2 * order + 1
        if ngridx % 2 == 0:
            ngridx += 1
    if ngridx <= 2 * order + 1:
        raise ValueError("The box must leave interior points between the boundary walls.")
    boxL_ang = ngridx * GRID_SPACING_ANG
    return boxL_ang, ngridx, GRID_SPACING_ANG


###############################################################################
# Laplacian Functions (Finite Difference Method)
###############################################################################

def laplacianCoeff(laplacian_order: int):
    """
    Compute finite-difference weights for approximating the second derivative (Laplacian) on a uniform grid
    using the method of undetermined coefficients (moment conditions).

    Constructs and solves the linear Vandermonde-like system moment_matrix weights = rhs, where
      A_{i,j} = (j - m)^i / i!, for i = 0..2m, j = 0..2m,
      and c corresponds to the second-derivative moment (rhs[2] = 2!).
    Only nonnegative offsets are returned; symmetry implies weights for negative offsets are identical.
    Complexity: O(m^3) for matrix inversion; suitable for moderate m.

    ref - Fornberg, B. Mathematics of Computation, 51(184), 699-706. (1988)

    input:
        int         laplacian_order  Accuracy (dimension) of the Laplacian operator
    output:
        np.array    laplacian_coeff     Coefficients
    """
    moment_matrix = np.zeros((2*laplacian_order+1, 2*laplacian_order+1))
    rhs = np.zeros(2*laplacian_order+1)
    rhs[2] = math.factorial(2)
    for i in range(2*laplacian_order+1):
        for j in range(2*laplacian_order+1):
            moment_matrix[i, j] = (j - laplacian_order)**i
    inv_moment_matrix = lin.inv(moment_matrix)
    weights = np.matmul(inv_moment_matrix, rhs)
    laplacian_coeff = weights[laplacian_order:2*laplacian_order+1]
    return laplacian_coeff

def laplacian(ngridx, laplacian_order, dx):
    """
    Construct the discrete Laplacian operator matrix on a periodic one-dimensional grid.

    The operator uses a central finite-difference stencil of width 2m+1,
    where m = laplacian_order

    ref - Fornberg, B. Mathematics of Computation, 51(184), 699–706. (1988)

    input:
        int         ngridx          Number of spatial grid points
        int         laplacian_order  Accuracy (dimension) of the Laplacian operator
        float       dx              Grid spacing
    output:
        csc_matrix  laplacian_op    Sparse Laplacian operator matrix
    """
    laplacian_coeff = laplacianCoeff(laplacian_order)
    # Keep the existing periodic stencil; potential() supplies the boundary walls.
    rows = np.repeat(np.arange(ngridx), 2 * laplacian_order + 1)
    offsets = np.tile(np.arange(-laplacian_order, laplacian_order + 1), ngridx)
    columns = (rows + offsets) % ngridx
    values = laplacian_coeff[np.abs(offsets)] / dx**2
    return coo_matrix((values, (rows, columns)), shape=(ngridx, ngridx)).tocsc()

###############################################################################
# Potential Construction
###############################################################################

def potential(ngridx, pot_shape=0, pot_height_har=25, curvature=0.5, width=0, slope=0.5, distance=0, dx=0, boundary_points=SIM_CFG.laplacian_order):
    """
    Set the shape of potential V and return the potential on a grid.

    input:
        int         ngridx              Number of spatial grid points
        int         pot_shape           Shape of potential:
                                         0 = No potential (Box)
                                         1 = Step potential
                                         2 = Single wall
                                         3 = Double wall
                                         4 = Finite well (packet starts at middle)
                                         5 = Harmonic well
        int         pot_height_har       Height of the potential barrier (Hartree)
        int         barrier_thicknss_bohr    Thickness of the potential barrier (Bohr)
    output:
        np.array    pot_grid            Potential on the grid
    """

    # pot_height_har = pot_height_eV / effective_har2ev

    # Initialize potential grid
    pot_grid = np.zeros(ngridx)
    pot_grid[0] = 1e9
    pot_grid[ngridx-1] = 1e9

    width    = int(width/dx)
    center   = int(ngridx * 0.5)
    distance = int(distance/dx)
    

    if pot_shape == 0:      # Finite square well
        '''
                     width
                     <---->
        -----------─┐     ┌------------ height
                    │     │     
                    │     │      
                    └-----┘             0
        ---------------┼--------------->
                    center
        '''
        pot_grid[1:ngridx] = pot_height_har
        pot_grid[center - width // 2 : center + width // 2] = 0

    elif pot_shape == 1:    # infinite square well
        '''
                    │     │
                    │width│             ↑ inf
                    │<--->│
                    │     │
                    │     │     
                    │     │     
                    └-----┘             0
        ---------------┼--------------->
                    center
        '''
        pot_grid[1:ngridx] = 1e9
        pot_grid[center - width // 2 : center + width // 2] = 0    

    elif pot_shape == 2:    # Symmetric double square well
        '''
              width distance width
              <----><------><---->
        -----─┐     ┌-------┐     ┌----- height
              │     │       │     │
              │     │       │     │
              └-----┘       └-----┘      0
        ----------------┼---------------->
                     center
               
        '''
        pot_grid[1:ngridx] = pot_height_har
        pot_grid[center - width - distance // 2: center - distance // 2] = 0
        pot_grid[center + distance // 2: center + width + distance // 2] = 0

    elif pot_shape == 3:    # Harmonic
        '''
        y=a(x-b)^2

        curvature = y'' = 2a
        '''
        for i in range(1, ngridx):
            pot_grid[i] = (i - (ngridx-1)//2)**2 / (((ngridx-1)//2)**2) # normalize
        pot_grid[1:ngridx] = pot_grid[1:ngridx] * pot_height_har * curvature

    elif pot_shape == 4:    # Triangular
        '''
            │   /
            │  /
            │ /
            │/
        ----┼---------------->
           vertex
        '''
        center = int(ngridx * 0.5)
        tri_width = max(2, width)
        vertex = center - tri_width // 2
        left = max(1, vertex)
        right = min(ngridx - 1, vertex + tri_width)
        pot_grid[:] = 1e6
        for i in range(0, max(0, right - left)):
            pot_grid[left + i] = i * dx * slope

    elif pot_shape == 5:    # Asymmetric potential (1)
        '''
                 │width
                 │<---->
                 │     ┌---------- height
                 │     │
                 │     │
                 └-----┘           0
        ------------┼--------------->
                 center
        '''

        pot_grid[1 : width] = 0  
        pot_grid[width: ngridx - 1] = pot_height_har  
    
    elif pot_shape == 6:    # Asymmetric potential (2)
        '''
           │
           │width distance width
           │<----><------><---->
           │     ┌-------┐     ┌----- height
           │     │       │     │
           │     │       │     │
           └-----┘       └-----┘      0
        -------------┼---------------->
                 center
        '''

        pot_grid[1 : width] = 0  
        pot_grid[width : width + distance] = pot_height_har 
        pot_grid[width + distance : 2*width + distance] = 0
        pot_grid[2*width + distance: ngridx - 1] = pot_height_har 

    elif pot_shape == 7:    # Custum potential
        pot_grid[1 : ngridx - 1] = 0  

    # Apply boundary walls last to avoid overwriting them with the potential shape.
    # Cover the finite-difference stencil using large finite walls, not np.inf.
    if not 1 <= boundary_points < ngridx / 2:
        raise ValueError("Boundary walls must leave interior grid points.")
    pot_grid[:boundary_points] = BOUNDARY_WALL_HAR
    pot_grid[-boundary_points:] = BOUNDARY_WALL_HAR
    return pot_grid

###############################################################################
# Hamiltonian and TISE Solver
###############################################################################

def hamiltonian(ngridx, laplacian_order, pot_height_har=25, pot_shape=0, 
                curvature=0.5, width=0, slope=0.6, distance=0, dx=0, boundary_points=SIM_CFG.laplacian_order):
    """
    Define the Hamiltonian using the Laplacian and Potential operators.

    H = - (1 / 2) ∇^2 + V

    """
    # Setup potential operator
    potential_grid = potential(ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx, boundary_points=laplacian_order)
    pot_op = diags(potential_grid, format='csc')

    # Setup Laplacian operator
    laplacian_op = laplacian(ngridx, laplacian_order, dx)

    # Setup Hamiltonian operator with effective mass
    ham = - 1.0 * laplacian_op / 2. + pot_op
    return ham

def state_energy_cutoff(potential_grid, pot_shape, pot_height_har):
    """Use the outer barrier for finite wells, and retain other display cutoffs."""
    if pot_shape in (0, 2, 5, 6):
        return pot_height_har  # Bound states only: E < the exterior barrier.
    if pot_shape == 4:
        return 1e6            # Exclude states in the artificial outer plateau.
    if pot_shape == 1:
        return 1e2 * pot_height_har
    return 1.5 * pot_height_har


def solve_tise(pot_shape=0, curvature=0.5, width=0, slope=0.6, distance=0, 
               pot_height_har=25, dx=0, ngridx=SIM_CFG.ngridx,
               laplacian_order=SIM_CFG.laplacian_order, num_state=5):
    """Return up to num_state lowest eigenpairs for the existing display."""
    if (not isinstance(num_state, (int, np.integer)) or isinstance(num_state, bool)
            or not 1 <= num_state < ngridx - 1):
        raise ValueError("num_state must be an integer from 1 to ngridx - 2.")
    if (not np.isfinite(dx) or dx <= 0 or laplacian_order < 1
            or ngridx <= 2 * laplacian_order + 1):
        raise ValueError("Use positive grid spacing and a grid wider than the stencil.")
    print("\n(1) Solving time-independent Schrodinger equation...")
    ham_op = hamiltonian(ngridx, laplacian_order, pot_height_har, pot_shape, 
                         curvature, width, slope, distance, dx,)
    # The kinetic operator is positive semidefinite, so min(V)-1 lies below H.
    # Shift-invert finds only the requested lowest states, not the full spectrum.
    # A real symmetric matrix avoids the complex ARPACK path used by Bloch problems.
    potential_grid = potential(ngridx, pot_shape, pot_height_har, curvature,
                               width, slope, distance, dx, boundary_points=laplacian_order)
    # An upper energy filter is already used by compute_energy() and animate().
    # Since T >= 0, the number of eigenvalues below that cutoff cannot exceed
    # the number of potential-grid entries below it (the min-max principle).
    # In a narrow infinite well, this avoids requesting huge wall-energy states.
    energy_cutoff = state_energy_cutoff(potential_grid, pot_shape, pot_height_har)
    count = min(num_state, int(np.count_nonzero(potential_grid < energy_cutoff)))
    if count == 0:
        print("\nDone!! No states can pass the existing energy filter.")
        return np.empty(0), np.empty((ngridx, 0))
    # Keep the existing separate shifts for strongly separated potential groups.
    # One shift can be inaccurate across a large energy gap. Split only when
    # a kinetic-norm bound separates the groups.
    coeff = laplacianCoeff(laplacian_order)
    kinetic_bound = (abs(coeff[0]) + 2 * np.sum(np.abs(coeff[1:]))) / (2 * dx**2)
    ordered_potential = np.sort(potential_grid)[:count]
    splits = [0] + (np.flatnonzero(np.diff(ordered_potential)
                                  > 2 * kinetic_bound + 2) + 1).tolist() + [count]
    for start, stop in zip(splits[1:-1], splits[2:]):
        gap = ordered_potential[start] - ordered_potential[start - 1]
        span = ordered_potential[stop - 1] - ordered_potential[start]
        if gap <= span + 2 * kinetic_bound + 2:
            splits = [0, count]
            break
    # Sorted eigenvalues satisfy V_j <= E_j <= V_j + ||T||.
    # The checks above ensure each group's requested states are closer to its
    # shift than any lower group; no state is removed by this separate solve.
    values, vectors = [], []
    for start, stop in zip(splits[:-1], splits[1:]):
        group_count = stop - start
        sigma = float(ordered_potential[start]) - 1.0
        group_values, group_vectors = eigsh(
            ham_op, k=group_count, sigma=sigma, which='LM', tol=1e-10,
            ncv=min(ngridx - 1, max(4 * group_count + 1, 40)),
            v0=np.random.default_rng(20260917).normal(size=ngridx))
        values.append(group_values)
        vectors.append(group_vectors)
    eigval, eigvec = np.concatenate(values), np.column_stack(vectors)
    # Apply the same permutation to energies and wavefunction columns.
    order = np.argsort(eigval)
    eigval, eigvec = eigval[order], eigvec[:, order]
    print("\nDone!!")
    return eigval, eigvec

###############################################################################
# Energy Analysis
###############################################################################

def compute_energy(eigval, eigvec, num_state, pot_shape, curvature, width, slope, 
                   distance, pot_height_har, dx, ngridx=SIM_CFG.ngridx, 
                   laplacian_order=SIM_CFG.laplacian_order, effective_har2ev=har2ev):
    '''
    T_op = - (1 / 2) laplacian,  V_op = diag(potential)
    '''
    kinetic_op = -laplacian(ngridx, laplacian_order, dx) / 2.0
    potential_grid = potential(ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx, boundary_points=laplacian_order)
    potential_op = diags(potential_grid, format='csc')

    energy_cutoff = state_energy_cutoff(potential_grid, pot_shape, pot_height_har)
    filtered_indices = np.flatnonzero(eigval < energy_cutoff).tolist()

    num_to_print = min(num_state, len(filtered_indices))
    print("\nEnergy components for eigenstates :")
    for idx in range(num_to_print):
        j = filtered_indices[idx]
        psi = eigvec[:, j]
        kinetic_exp = np.vdot(psi, kinetic_op @ psi).real
        potential_exp = np.vdot(psi, potential_op @ psi).real
        total = kinetic_exp + potential_exp
        print(f"State {j+1}: Kinetic = {kinetic_exp * effective_har2ev:.2f} eV, Potential = {potential_exp * effective_har2ev:.2f} eV, Total = {total * effective_har2ev:.3f} eV")



# (2) Visualization & interactive widget

In [2]:

###############################################################################
# Animation and Visualization
###############################################################################

def state_plot_ranges(x, potential_ev, eigvec, indices, energies_ev, scale,
                      pot_shape, energy_cutoff_ev, boundary_points):
    """Frame the displayed states, using one range throughout the animation."""
    interior = np.zeros(len(x), dtype=bool)
    interior[boundary_points:-boundary_points] = True
    well_region = interior & (potential_ev < energy_cutoff_ev)
    x_bounds = []
    y_bounds = []
    for index, energy in zip(indices, energies_ev):
        psi = eigvec[:, index]
        norm = np.sum(np.abs(psi)**2)
        probability = np.abs(psi)**2 / norm
        cumulative = np.cumsum(probability)
        # Include 99.9% of each state's probability, then add visible space.
        left = min(int(np.searchsorted(cumulative, 0.0005)), len(x) - 1)
        right = min(int(np.searchsorted(cumulative, 0.9995)), len(x) - 1)
        x_bounds.extend((x[left], x[right]))
        # Real/imaginary parts can reach either sign at any animation time.
        amplitude = np.max(np.abs(psi)) / norm * abs(scale)
        density_height = np.max(probability) * abs(scale) * 6
        y_bounds.extend((energy - amplitude,
                         energy + max(amplitude, density_height)))
    # Keep all wells in view even when a selected state occupies only one well.
    if pot_shape in (0, 1, 2, 4, 5, 6) and np.any(well_region):
        x_bounds.extend((x[well_region][0], x[well_region][-1]))
    if not x_bounds:
        x_bounds.extend((x[interior][0], x[interior][-1]))
    lower_x, upper_x = min(x_bounds), max(x_bounds)
    x_margin = max(2.0, 0.05 * (upper_x - lower_x))
    dx_ang = x[1] - x[0]
    x_range = [max(x[0] - dx_ang / 2, lower_x - x_margin),
               min(x[-1] + dx_ang / 2, upper_x + x_margin)]
    visible = interior & (x >= x_range[0]) & (x <= x_range[1])
    if np.any(visible):
        # Include the potential bottom, without scaling to enormous outer walls.
        y_bounds.append(float(np.min(potential_ev[visible])))
    if not indices:
        low_potential = potential_ev[visible & well_region]
        if len(low_potential):
            y_bounds.extend((float(np.min(low_potential)), float(np.max(low_potential))))
        if pot_shape in (0, 2, 5, 6):
            y_bounds.append(energy_cutoff_ev)
    lower_y, upper_y = min(y_bounds), max(y_bounds)
    y_margin = max(0.05, 0.05 * (upper_y - lower_y))
    y_range = [lower_y - y_margin, upper_y + y_margin]
    return x_range, y_range


# Animation Function: Animate the time evolution of multiple eigenstates' real, imaginary, and probability density
# Only animate eigenstates with eigenenergy below the potential height.
def animate(eigval, eigvec, num_state, boxL, ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx, sim_time, effective_ang2bohr=ang2bohr, effective_har2ev=har2ev, boundary_points=SIM_CFG.laplacian_order):
    print("\n(2) Visualizing ...")
    print("\tRed \t= Real part of Psi \n\tBlue \t= Imaginary part of Psi \n\tGrey shaded = Probability Psi*Psi")

    # Define the x-axis in Angstrom units
    x = (np.arange(ngridx) - (ngridx - 1) / 2) * dx / effective_ang2bohr
    pot = potential(ngridx, pot_shape, pot_height_har, curvature, width, slope, distance, dx, boundary_points=boundary_points) * effective_har2ev
    scale = 0.8 * pot_height_har * effective_har2ev  # eV, coefficient for visualization

    # Apply the same state selection as the solver and energy analysis.
    energy_cutoff = state_energy_cutoff(pot / effective_har2ev, pot_shape, pot_height_har)
    filtered_indices = np.flatnonzero(eigval < energy_cutoff).tolist()
    num_to_plot = min(num_state, len(filtered_indices))

    # Compute normalization and energy offsets for each filtered eigenstate
    len_psi_list = [np.sum(np.abs(eigvec[:, j])**2) for j in filtered_indices[:num_to_plot]]
    E_eV_list = [eigval[j] * effective_har2ev for j in filtered_indices[:num_to_plot]]
    # Convert displayed energies to Joule; effective-mass scaling is already included.
    E_J_list = np.asarray(E_eV_list) * ev_energy

    x_range, y_range = state_plot_ranges(
        x, pot, eigvec, filtered_indices[:num_to_plot], E_eV_list, scale,
        pot_shape, energy_cutoff * effective_har2ev, boundary_points)

    sim_time_s = sim_time * 1e-15  # Convert fsec to seconds
    time_step = 2e-17              # 0.02 fsec
    t_values = np.arange(0, sim_time_s + 1e-20, time_step)

    def _lightness_by_rank(idx, n, low=35.0, high=70.0):
        if n <= 1:
            return 0.5 * (low + high)
        return low + (high - low) * (idx / (n - 1))

    real_colors = [
        f"hsl(6, 90%, {_lightness_by_rank(i, num_to_plot):.1f}%)"
        for i in range(num_to_plot)
    ]
    imag_colors = [
        f"hsl(215, 88%, {_lightness_by_rank(i, num_to_plot):.1f}%)"
        for i in range(num_to_plot)
    ]
    fill_colors = [
        f"rgba(120,120,120,{(0.24 + 0.18 * (0.0 if num_to_plot <= 1 else i / (num_to_plot - 1))):.3f})"
        for i in range(num_to_plot)
    ]

    def _state_curves(state_idx, t):
        j = filtered_indices[state_idx]
        phase = np.exp(-1j * E_J_list[state_idx] * t / hbar)
        psi_t = eigvec[:, j] * phase

        real_scaled = psi_t.real / len_psi_list[state_idx] * scale + E_eV_list[state_idx]
        imag_scaled = psi_t.imag / len_psi_list[state_idx] * scale + E_eV_list[state_idx]
        prob_scaled = ((psi_t.real)**2 + (psi_t.imag)**2) / len_psi_list[state_idx] * (scale * 6) + E_eV_list[state_idx]

        baseline = np.full_like(x, E_eV_list[state_idx])
        x_fill = np.concatenate([x, x[::-1]])
        y_fill = np.concatenate([prob_scaled, baseline[::-1]])
        return real_scaled, imag_scaled, x_fill, y_fill

    static_annotations = []
    x_text = x_range[0] + 0.05 * (x_range[1] - x_range[0])
    label_offset = 0.025 * (y_range[1] - y_range[0])
    for energy in E_eV_list:
        static_annotations.append(
            dict(
                x=x_text,
                y=energy + label_offset,
                text=f"{energy:.2f} eV",
                showarrow=False,
                font=dict(size=10, color="black"),
                xanchor="left",
                yanchor="middle",
            )
        )

    initial_data = [
        go.Scatter(
            x=x,
            y=pot,
            mode="lines",
            name="Potential",
            line=dict(color="black", width=1.2),
        )
    ]

    for idx in range(num_to_plot):
        real_scaled, imag_scaled, x_fill, y_fill = _state_curves(idx, t_values[0])
        initial_data.append(
            go.Scatter(
                x=x,
                y=real_scaled,
                mode="lines",
                name=f"Re(Psi) state {idx + 1}",
                line=dict(color=real_colors[idx], width=1.5),
            )
        )
        initial_data.append(
            go.Scatter(
                x=x,
                y=imag_scaled,
                mode="lines",
                name=f"Im(Psi) state {idx + 1}",
                line=dict(color=imag_colors[idx], width=1.5),
            )
        )
        initial_data.append(
            go.Scatter(
                x=x_fill,
                y=y_fill,
                mode="lines",
                name=f"|Psi|^2 state {idx + 1}",
                line=dict(color="rgba(0,0,0,0)", width=0),
                fill="toself",
                fillcolor=fill_colors[idx],
            )
        )

    frames = []
    dynamic_trace_indices = list(range(1, 1 + 3 * num_to_plot))
    for iframe, t in enumerate(t_values):
        frame_data = []
        for idx in range(num_to_plot):
            real_scaled, imag_scaled, x_fill, y_fill = _state_curves(idx, t)
            frame_data.append(
                go.Scatter(
                    x=x,
                    y=real_scaled,
                    mode="lines",
                    line=dict(color=real_colors[idx], width=1.5),
                )
            )
            frame_data.append(
                go.Scatter(
                    x=x,
                    y=imag_scaled,
                    mode="lines",
                    line=dict(color=imag_colors[idx], width=1.5),
                )
            )
            frame_data.append(
                go.Scatter(
                    x=x_fill,
                    y=y_fill,
                    mode="lines",
                    line=dict(color="rgba(0,0,0,0)", width=0),
                    fill="toself",
                    fillcolor=fill_colors[idx],
                )
            )

        frames.append(
            go.Frame(
                data=frame_data,
                traces=dynamic_trace_indices,
                name=str(iframe),
                layout=go.Layout(title_text=f"Time Evolution of Eigenstates (Time: {t * 1e15:.2f} fsec)"),
            )
        )

    slider_steps = []
    for iframe, t in enumerate(t_values):
        slider_steps.append(
            {
                "args": [[str(iframe)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
                "label": f"{t * 1e15:.2f}",
                "method": "animate",
            }
        )

    fig = go.Figure(data=initial_data, frames=frames)
    fig.update_layout(
        width=920,
        height=700,
        template="plotly_white",
        title=f"Time Evolution of Eigenstates (Time: {t_values[0] * 1e15:.2f} fsec)",
        margin=dict(l=80, r=220, t=90, b=140),
        xaxis=dict(title="Box [Angstrom]", range=x_range),
        yaxis=dict(title="Energy [eV]", range=y_range),
        annotations=static_annotations,
        updatemenus=[
            {
                "type": "buttons",
                "direction": "left",
                "showactive": True,
                "x": 0.0,
                "y": -0.06,
                "xanchor": "left",
                "yanchor": "top",
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [None, {"fromcurrent": True, "frame": {"duration": 50, "redraw": True}, "transition": {"duration": 0}}],
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [[None], {"mode": "immediate", "frame": {"duration": 0, "redraw": False}, "transition": {"duration": 0}}],
                    },
                ],
            }
        ],
        sliders=[
            {
                "active": 0,
                "x": 0.22,
                "y": -0.06,
                "len": 0.76,
                "currentvalue": {"prefix": "Time [fsec]: "},
                "pad": {"t": 40},
                "steps": slider_steps,
            }
        ],
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1.0,
            xanchor="left",
            x=1.02,
            bgcolor="rgba(255,255,255,0.85)",
        ),
    )

    # Time controls are only useful when the simulation duration is positive.
    for menu in fig.layout.updatemenus:
        menu.visible = sim_time > 0
    for slider in fig.layout.sliders:
        slider.visible = sim_time > 0

    display(HTML(fig.to_html(full_html=False, include_plotlyjs='inline', auto_play=False)))

    print("\n")
    print("Done!!")



###############################################################################
# Interactive Widgets
###############################################################################

if __name__=="__main__":
    ### Calculation Setting
    # The actual grid is set from the current well parameters inside main().
    laplacian_order = SIM_CFG.laplacian_order
    print(f'Box padding = {BOX_PADDING_ANG:g} Angstrom; grid spacing = {GRID_SPACING_ANG:.3f} Angstrom')


    ### Widget
    # Widget setting
    widget_width    = 400
    widget_margin   = 5
    widget_text     = 200
    
    # Widget(1): Calculation
    shape_widget = widgets.Dropdown(
        options=[
            ('Finite square well',          0),
            ('Infinite square well',        1),
            ('Double square well',          2),
            ('Harmonic oscillator',         3),
            ('Triangular well',             4),
            ('Asymmetric square well (1)',  5),
            ('Asymmetric square well (2)',  6),
            ('Custum potential',            7),
        ],
        value=0, description='Potential Shape', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    m_eff_widget = widgets.Dropdown(
        options=[
            ('GaAs : 0.07',  0.07),
            ('Silicon : 1.08', 1.08),
            ('Germanium : 0.55', 0.55),
            ('Bare mass : 1.00',  1.00),
        ],
        value=1.00, description='Effective mass', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    width_widget = widgets.FloatSlider(
        min=1, max=100, step=0.1,value=15, description='Width [Ang]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    height_widget = widgets.FloatSlider(
        value=4, min=0.2, max=30, step=0.1,  description='Height [eV]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    curvature_widget = widgets.FloatSlider(
        min=0.1, max=10, step=0.1, value=1, description='Curvature of harmonic well',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    slope_widget = widgets.FloatSlider(
        min=0.1, max=50, step=0.1, value=3, description='Slope of triangular well [eV/Ang]',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    distance_widget = widgets.FloatSlider(
        min=0, max=100, step=0.2,value=5, description='Distance [Ang]', 
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )

    # Widget(2): Visualization
    num_state_widget = widgets.IntSlider(
        min=1, max=10, step=1, value=5, description='# of States',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{5*widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    sim_time_widget = widgets.IntSlider(
        min=0, max=10, step=1, value=0, description='Simulation Time [fsec]',
        readout=True, style={'description_width': f'{widget_text}px'}, layout=widgets.Layout(width=f'{widget_width}px', margin=f'{widget_margin}px {widget_margin}px {widget_margin}px {widget_margin}px')
    )
    
    # Real time visibility control
    def update_visibility(change):
        value = shape_widget.value
        width_widget.layout.display     = 'flex'   if value <= 2 or value >= 4 else 'none'
        distance_widget.layout.display  = 'flex'   if value == 2 or value == 6 else 'none'
        height_widget.layout.display    = 'flex'   if value == 0 or value == 2 or value >= 5 else 'none'
        curvature_widget.layout.display = 'flex'   if value == 3 else 'none'
        slope_widget.layout.display     = 'flex'   if value == 4 else 'none'

    curvature_widget.layout.display = 'none'
    distance_widget.layout.display  = 'none'
    # width_widget.layout.display   = 'none'
    # height_widget.layout.display  = 'none'
    slope_widget.layout.display     = 'none'
    sim_time_widget.layout.display = 'none'
    shape_widget.observe(update_visibility, names='value')
    update_visibility(None)

    @interact_manual(
        pot_shape = shape_widget,
        m_eff     = m_eff_widget,
        curvature = curvature_widget,
        width     = width_widget,
        slope     = slope_widget,
        pot_height_eV = height_widget,
        distance  = distance_widget,
        

        num_state = num_state_widget,
        sim_time  = sim_time_widget
    )
    def main(pot_shape, m_eff, curvature, width, slope, pot_height_eV, distance, 
             num_state, sim_time):
        
        ### Unit conversion
        '''
        Effective mass atomic unit

        - ref
            Quantum Mechanics with Applications to Nanotechnology and Information Science, Yehuda B. Band, Yshai Avishai, Ch3 (2013)
            https://www.sciencedirect.com/topics/mathematics/atomic-unit

 
        - Units
            Effective (mass) Bohr radius
                a*   = a0 / m*        (a0 is Bohr radius)

            Effective (mass) Hartree
                E_h* = m* x E_h    (E_h is Hartree energy)

        - Schrodinger equation

            iℏ * ∂​Ψ/∂t(r,t) = ( -( ℏ^2 / 2m* ) ​​∇^2 + V(r,t) ) Ψ(r,t)

                ↓ 

            i * ∂​Ψ/∂t(r,t) = ( -( 1 / 2 ) ​​∇^2 + V(r,t) ) Ψ(r,t)

        '''
        
        effective_bohr = bohr_radius / m_eff
        effective_ang2bohr = angstrom_radius / effective_bohr
        effective_har = hartree_energy * m_eff
        effective_har2ev = effective_har / ev_energy
        
        ### Grid setting
        boxL_ang, ngridx, dx_ang = simulation_grid(pot_shape, width, distance, m_eff)
        print('(0) Grid setting')
        print(f'box length \t = {boxL_ang:.3f} Angstrom')
        print(f'number of x grid = {ngridx}')
        print(f'dx \t\t = {dx_ang:.3f} Angstrom')

        dx              = dx_ang    * effective_ang2bohr
        boxL            = boxL_ang  * effective_ang2bohr
        pot_height_har  = pot_height_eV / effective_har2ev
        width_bohr      = width     * effective_ang2bohr
        distance_bohr   = distance  * effective_ang2bohr
        slope_har       = slope     / (effective_har2ev * effective_ang2bohr)    # [Hartree/Bohr] from [eV/Ang]

        
        ### Calculation & Visualization
        (eigval, eigvec) = solve_tise(pot_shape, curvature, width_bohr, slope_har, distance_bohr, pot_height_har, dx, ngridx, laplacian_order, num_state=num_state)

        compute_energy(eigval, eigvec, num_state, 
                       pot_shape, curvature, width_bohr, slope_har, distance_bohr, pot_height_har, dx, ngridx, laplacian_order, effective_har2ev)
        
        animate(eigval, eigvec, num_state, 
                boxL, ngridx, pot_shape, pot_height_har, curvature, width_bohr, slope_har, distance_bohr, dx, 
                sim_time, effective_ang2bohr, effective_har2ev, boundary_points=laplacian_order)

    readme = r"""
              width
       ↑      <---->
height ┼      ┌-----┐
       │      │     │
       │      │     │
       │------┘     └------
       --------------------->

    """
    print(readme)
    pass


Box padding = 80 Angstrom; grid spacing = 0.200 Angstrom


interactive(children=(Dropdown(description='Potential Shape', layout=Layout(margin='5px 5px 5px 5px', width='4…


              width
       ↑      <---->
height ┼      ┌-----┐
       │      │     │
       │      │     │
       │------┘     └------
       --------------------->

    
